# AutoML · M01: Baselines

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | AutoML — Pre-Modelado |
| **Módulo** | M01 — Baselines |

---

## 🎯 Qué hace

Entrena modelos baseline (DummyClassifier, LogisticRegression, RandomForest,
GradientBoosting, XGBoost) sobre el dataset canónico de producción
`dataset_final_tfm.parquet` (24 features, 33.621 expedientes) para establecer
una referencia de rendimiento antes del screening AutoML completo.

Cada modelo se evalúa con validación cruzada estratificada (CV=5) y con un
split de test independiente (70/30). Se reportan métricas globales y por clase.

## 📋 Requisitos

- `data/automl/dataset_final_tfm.parquet` — 33.621 × 25 (generado por f3_m05)
- Entorno: `tfm_abandono`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/results_baselines.parquet` | Métricas de los modelos baseline |
| `docs/html/fase_automl/m01_baselines.html` | Informe HTML |

## 🔄 Flujo

```
data/automl/dataset_final_tfm.parquet (24 features + abandono)
    ↓ DummyClassifier, LogisticRegression, RandomForest,
      GradientBoosting, XGBoost
    ↓ CV=5 estratificado + test 30%
    ↓ Métricas: F1, AUC, Precision, Recall (global + por clase)
    → results_baselines.parquet + HTML
```

## ➡️ Siguiente

`fautoml_m02_lazypredict.ipynb` — screening rápido de ~30 clasificadores

---

> **Nota metodológica:** En la versión anterior del dataset (20 features, sin
> `tasa_abandono_titulacion`), XGBoost alcanzaba F1=0.7821 y AUC=0.9288.
> El dataset actual incorpora 6 nuevas variables validadas en Fase 3,
> siendo `tasa_abandono_titulacion` (target encoding de titulación) la de
> mayor poder predictivo esperado. Los resultados de este módulo confirman
> el impacto de esas features sobre el baseline.


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Detecta ROOT subiendo niveles hasta encontrar src/.
# Importa config, utils y html desde src — sin hardcoding de rutas.
# ============================================================================

import sys
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Detectar ROOT robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

# Imports del proyecto
from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import generar_kpis_html, generar_seccion_html
from src.html.render import render_pagina

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, log_loss
)

# XGBoost opcional
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('⚠️  XGBoost no disponible')

# Rutas y constantes
RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])

fmt           = formato_numero_es
RANDOM_STATE  = 42
CV_FOLDS      = 5
TEST_SIZE     = 0.30
FRAMEWORK     = 'baselines'
NOTEBOOK_NAME = 'fautoml_m01_baselines.ipynb'
CARPETA_NB    = 'fase_automl'

info_entorno()
print('\n✅ Configuración completada')


✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASET Y VERIFICACIÓN ANTI-LEAKAGE
# ============================================================================
# Dataset canónico: dataset_final_tfm.parquet (24 features + abandono).
# Generado por f3_m05_target_export.ipynb tras auditoría completa de leakage.
# Verificación: ninguna columna que forme parte de la definición del target.
# ============================================================================

print('\n' + '='*70)
print('CARGANDO DATASET CANÓNICO')
print('='*70)

ruta_dataset = RUTA_AUTOML / 'dataset_final_tfm.parquet'
df = pd.read_parquet(ruta_dataset)

TARGET   = 'abandono'
X        = df.drop(columns=[TARGET])
y        = df[TARGET]

n_total    = len(df)
n_features = len(X.columns)
n_abandono = int(y.sum())
tasa_ab    = n_abandono / n_total

print(f'\n✓ Dataset: {ruta_dataset.name}')
print(f'  Registros : {fmt(n_total)}')
print(f'  Features  : {n_features}')
print(f'  Abandono  : {fmt(n_abandono)} ({tasa_ab*100:.1f}%)')
print(f'  No abandona: {fmt(n_total - n_abandono)} ({(1-tasa_ab)*100:.1f}%)')

# Verificación anti-leakage — columnas prohibidas no deben existir
COLS_LEAKAGE = [
    'egresado', 'egresado_de_hecho', 'curso_ultimo',
    'anos_inactivo', 'pct_titulacion', 'cred_faltantes',
    'progreso_relativo', 'titulacion', 'per_id_ficticio', 'exp_tit_id'
]
leakage_presente = [c for c in COLS_LEAKAGE if c in X.columns]
if leakage_presente:
    print(f'\n🚨 LEAKAGE DETECTADO: {leakage_presente}')
    raise ValueError('Detener — dataset con leakage')
else:
    print('\n✅ Verificación anti-leakage superada')

print(f'\n📋 Features ({n_features}):')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')



CARGANDO DATASET CANÓNICO

✓ Dataset: dataset_final_tfm.parquet
  Registros : 33.621
  Features  : 24
  Abandono  : 9.833 (29.2%)
  No abandona: 23.788 (70.8%)

✅ Verificación anti-leakage superada

📋 Features (24):
   1. cred_superados_anio_1er
   2. nota_1er_anio
   3. nota_acceso
   4. nota_selectividad
   5. via_acceso
   6. orden_preferencia
   7. cupo
   8. tasa_abandono_titulacion
   9. rama
  10. cred_repetidos
  11. tasa_repeticion
  12. sexo
  13. edad_entrada
  14. pais_nombre
  15. provincia
  16. universidad_origen
  17. n_anios_beca
  18. anios_sin_beca
  19. situacion_laboral
  20. n_anios_trabajando
  21. max_pagos
  22. indicador_interrupcion
  23. anios_gap
  24. n_anios_sin_notas


In [3]:
# ============================================================================
# CELDA 3: SPLIT TRAIN/TEST Y DEFINICIÓN DE MODELOS
# ============================================================================
# Split estratificado 70/30 con semilla fija.
# SimpleImputer(median) en pipeline — robusto ante nulos residuales.
# Modelos seleccionados: 2 dummy (cota inferior) + 4 reales progresivos.
# ============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size     = TEST_SIZE,
    random_state  = RANDOM_STATE,
    stratify      = y
)

print(f'Split estratificado {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)}:')
print(f'  Train: {fmt(len(X_train))} ({(y_train==1).mean()*100:.1f}% abandono)')
print(f'  Test : {fmt(len(X_test))}  ({(y_test==1).mean()*100:.1f}% abandono)')

# Definición de modelos
MODELOS = [
    ('DummyClassifier_stratified',
     DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)),
    ('DummyClassifier_majority',
     DummyClassifier(strategy='most_frequent')),
    ('Regresion Logistica',
     LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)),
    ('Random Forest',
     RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    ('Gradient Boosting',
     GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)),
]
if HAS_XGB:
    MODELOS.append((
        'XGBoost',
        XGBClassifier(
            n_estimators=100, random_state=RANDOM_STATE,
            n_jobs=-1, eval_metric='logloss', verbosity=0
        )
    ))

print(f'\nModelos a evaluar: {len(MODELOS)}')
for nombre, _ in MODELOS:
    print(f'  · {nombre}')


Split estratificado 70/30:
  Train: 23.534 (29.2% abandono)
  Test : 10.087  (29.2% abandono)

Modelos a evaluar: 6
  · DummyClassifier_stratified
  · DummyClassifier_majority
  · Regresion Logistica
  · Random Forest
  · Gradient Boosting
  · XGBoost


In [4]:
# ============================================================================
# CELDA 4: ENTRENAR Y EVALUAR MODELOS
# ============================================================================
# Cada modelo se evalúa con:
#   - CV=5 estratificado sobre train (F1 macro como métrica principal)
#   - Split test independiente: métricas completas + reporte por clase
# Pipeline con SimpleImputer para robustez ante nulos residuales.
# ============================================================================

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
all_results = []

print('\n' + '='*70)
print(f'ENTRENANDO {len(MODELOS)} MODELOS (CV={CV_FOLDS})')
print('='*70)

for nombre, modelo in MODELOS:
    print(f'\n  [{len(all_results)+1}/{len(MODELOS)}] {nombre}...')
    t0 = time.time()

    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model',   modelo)
    ])

    # CV sobre train
    es_dummy = 'Dummy' in nombre
    if not es_dummy:
        scores_cv = cross_val_score(
            pipe, X_train, y_train,
            cv=cv, scoring='f1', n_jobs=-1
        )
        f1_cv_mean = scores_cv.mean()
        f1_cv_std  = scores_cv.std()
    else:
        f1_cv_mean = float('nan')
        f1_cv_std  = float('nan')

    # Fit final sobre train completo
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    try:
        y_prob = pipe.predict_proba(X_test)[:, 1]
    except Exception:
        y_prob = np.zeros(len(y_test))

    t_total = round(time.time() - t0, 2)

    # Métricas por clase (clase 1 = abandono)
    prec_clase1   = precision_score(y_test, y_pred, zero_division=0)
    recall_clase1 = recall_score(y_test, y_pred, zero_division=0)
    f1_clase1     = f1_score(y_test, y_pred, zero_division=0)

    resultado = {
        'framework'        : FRAMEWORK,
        'model_name'       : nombre,
        'accuracy'         : accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'precision'        : prec_clase1,
        'recall'           : recall_clase1,
        'f1'               : f1_clase1,
        'f1_cv_mean'       : f1_cv_mean,
        'f1_cv_std'        : f1_cv_std,
        'auc_roc'          : roc_auc_score(y_test, y_prob) if y_prob.sum() > 0 else 0,
        'mcc'              : matthews_corrcoef(y_test, y_pred),
        'kappa'            : cohen_kappa_score(y_test, y_pred),
        'log_loss'         : log_loss(y_test, y_prob) if y_prob.sum() > 0 else 999,
        'train_time_sec'   : t_total,
    }
    all_results.append(resultado)

    # Mostrar resumen
    print(f'    AUC={resultado["auc_roc"]:.4f} | F1={resultado["f1"]:.4f} | ', end='')
    if not es_dummy:
        print(f'F1_CV={f1_cv_mean:.4f}(±{f1_cv_std:.4f}) | {t_total}s')
    else:
        print(f'{t_total}s')

df_resultados = (
    pd.DataFrame(all_results)
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

print('\n--- RANKING FINAL (por F1 test) ---')
cols_show = ['model_name', 'f1', 'f1_cv_mean', 'f1_cv_std', 'auc_roc', 'mcc', 'train_time_sec']
print(df_resultados[cols_show].to_string(index=False))



ENTRENANDO 6 MODELOS (CV=5)

  [1/6] DummyClassifier_stratified...


    AUC=0.4908 | F1=0.2738 | 0.07s

  [2/6] DummyClassifier_majority...


    AUC=0.0000 | F1=0.0000 | 0.08s

  [3/6] Regresion Logistica...


    AUC=0.9084 | F1=0.7427 | F1_CV=0.7420(±0.0035) | 12.51s

  [4/6] Random Forest...


    AUC=0.9499 | F1=0.8205 | F1_CV=0.8210(±0.0016) | 9.49s

  [5/6] Gradient Boosting...


    AUC=0.9405 | F1=0.7911 | F1_CV=0.7906(±0.0042) | 16.56s

  [6/6] XGBoost...


    AUC=0.9532 | F1=0.8296 | F1_CV=0.8275(±0.0027) | 6.46s

--- RANKING FINAL (por F1 test) ---
                model_name       f1  f1_cv_mean  f1_cv_std  auc_roc       mcc  train_time_sec
                   XGBoost 0.829648    0.827543   0.002726 0.953174  0.764915            6.46
             Random Forest 0.820485    0.820992   0.001608 0.949932  0.753720            9.49
         Gradient Boosting 0.791074    0.790640   0.004195 0.940489  0.714654           16.56
       Regresion Logistica 0.742713    0.741963   0.003519 0.908425  0.647719           12.51
DummyClassifier_stratified 0.273779         NaN        NaN 0.490789 -0.018626            0.07
  DummyClassifier_majority 0.000000         NaN        NaN 0.000000  0.000000            0.08


In [5]:
# ============================================================================
# CELDA 5: GUARDAR RESULTADOS
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_baselines.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} modelos guardados')


💾 results_baselines.parquet: 6 modelos guardados


In [6]:
# ============================================================================
# CELDA 6: GENERAR HTML
# ============================================================================
# Tabla de resultados con métricas globales y por clase.
# Nota metodológica sobre evolución del dataset (20 cols → 24 cols).
# render_pagina infiere título, fase y módulo del nombre del notebook.
# ============================================================================

# --- KPIs ---
mejor     = df_resultados[~df_resultados['model_name'].str.startswith('Dummy')].iloc[0]
n_reales  = len(df_resultados[~df_resultados['model_name'].str.startswith('Dummy')])

KPIS = [
    {'valor': fmt(n_total),                   'titulo': 'Expedientes'},
    {'valor': str(n_features),                'titulo': 'Features'},
    {'valor': f'{mejor["f1"]:.4f}',           'titulo': f'Mejor F1 ({mejor["model_name"]})'},
    {'valor': f'{mejor["auc_roc"]:.4f}',      'titulo': f'Mejor AUC ({mejor["model_name"]})'},
]

# --- Tabla de resultados ---
cabecera_cols = [
    'Modelo', 'F1 test', 'F1 CV (media)', 'F1 CV (±std)',
    'AUC', 'Precision', 'Recall', 'MCC', 'Tiempo'
]
tabla = '<table style="width:100%;border-collapse:collapse;font-size:12px;">\n'
tabla += '<tr style="background:#3182ce;color:white;">'
for col in cabecera_cols:
    tabla += f'<th style="padding:8px;text-align:center;">{col}</th>'
tabla += '</tr>\n'

for i, row in df_resultados.iterrows():
    bg = '#f7fafc' if i % 2 == 0 else 'white'
    es_dummy = 'Dummy' in row['model_name']
    fw = 'normal' if es_dummy else 'bold'
    tabla += f'<tr style="background:{bg};">'
    nombre_modelo = row['model_name']
    tabla += f'<td style="padding:6px;font-weight:{fw}">{nombre_modelo}</td>'

    # F1 test
    v = row['f1']
    color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
    tabla += f'<td style="text-align:center;color:{color};font-weight:bold;">{v:.4f}</td>'

    # F1 CV
    if not es_dummy and not np.isnan(row['f1_cv_mean']):
        tabla += f'<td style="text-align:center;">{row["f1_cv_mean"]:.4f}</td>'
        tabla += f'<td style="text-align:center;">{row["f1_cv_std"]:.4f}</td>'
    else:
        tabla += '<td style="text-align:center;color:#a0aec0;">—</td>'
        tabla += '<td style="text-align:center;color:#a0aec0;">—</td>'

    # AUC
    v = row['auc_roc']
    color = '#38a169' if v > 0.85 else '#ed8936' if v > 0.6 else '#e53e3e'
    tabla += f'<td style="text-align:center;color:{color};">{v:.4f}</td>'

    # Precision, Recall, MCC
    for campo in ['precision', 'recall', 'mcc']:
        tabla += f'<td style="text-align:center;">{row[campo]:.4f}</td>'

    tabla += f'<td style="text-align:center;">{row["train_time_sec"]:.1f}s</td>'
    tabla += '</tr>\n'

tabla += '</table>'

s_tabla = generar_seccion_html(
    'Resultados por modelo — métricas globales y por clase (abandono=1)',
    tabla, '📊'
)

# --- Nota metodológica ---
nota_metodologica = generar_seccion_html(
    'Nota metodológica — evolución del dataset',
    '''
<div style="background:#f0f4f8;padding:18px;border-radius:10px;
            border-left:4px solid #3182ce;line-height:1.7;">
  <p><strong>Dataset de producción:</strong> <code>dataset_final_tfm.parquet</code>
  — 24 variables auditadas + target <code>abandono</code>.</p>
  <p><strong>Evolución respecto a la versión anterior</strong> (20 features, sin
  target encoding de titulación): XGBoost alcanzaba F1=0.7821 y AUC=0.9288.
  El dataset actual incorpora 6 nuevas variables validadas en Fase 3
  —<code>tasa_abandono_titulacion</code>, <code>cred_repetidos</code>,
  <code>tasa_repeticion</code>, <code>n_anios_trabajando</code>,
  <code>n_anios_sin_notas</code>, <code>cupo</code>—
  y elimina 2 variables cuestionables (<code>tuvo_beca</code>,
  <code>pago_fraccionado</code>).</p>
  <p><strong>Por qué un solo dataset:</strong> El dataset anterior (Caso D, 22 cols)
  incluía variables descartadas por riesgo de leakage en la auditoría de Fase 3.
  Comparar con él sería metodológicamente inconsistente. Los resultados de este
  módulo son la referencia honesta y reproducible del proyecto.</p>
  <p><strong>Validación cruzada:</strong> Los modelos no Dummy se evalúan con
  CV=5 estratificado sobre train. La columna <em>F1 CV (media ± std)</em> mide
  la estabilidad del modelo — alta std indica sensibilidad al particionado.</p>
</div>''',
    'ℹ️'
)

# --- Render y guardar HTML ---
ruta_html = RUTA_FASE_AUTOML_HTML / 'm01_baselines.html'

render_pagina(
    NOTEBOOK_NAME,
    generar_kpis_html(KPIS) + s_tabla + nota_metodologica,
    ruta_html,
    carpeta_notebook=CARPETA_NB
)

print(f'\n✅ HTML: {ruta_html}')



✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m01_baselines.html


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================

mejor_real = df_resultados[~df_resultados['model_name'].str.startswith('Dummy')].iloc[0]

print('\n' + '='*60)
print('✅ AUTOML-M01 COMPLETADO')
print('='*60)
print(f'Dataset       : dataset_final_tfm.parquet ({fmt(n_total)} × {n_features+1})')
print(f'Modelos       : {len(df_resultados)}')
print(f'Mejor modelo  : {mejor_real["model_name"]}')
print(f'  F1 test     : {mejor_real["f1"]:.4f}')
print(f'  F1 CV       : {mejor_real["f1_cv_mean"]:.4f} (±{mejor_real["f1_cv_std"]:.4f})')
print(f'  AUC         : {mejor_real["auc_roc"]:.4f}')
print(f'  MCC         : {mejor_real["mcc"]:.4f}')
print(f'Resultados    : {RUTA_AUTOML / "results_baselines.parquet"}')
print(f'HTML          : {RUTA_FASE_AUTOML_HTML / "m01_baselines.html"}')
print(f'\n➡️  Siguiente  : fautoml_m02_lazypredict.ipynb')
print('='*60)



✅ AUTOML-M01 COMPLETADO
Dataset       : dataset_final_tfm.parquet (33.621 × 25)
Modelos       : 6
Mejor modelo  : XGBoost
  F1 test     : 0.8296
  F1 CV       : 0.8275 (±0.0027)
  AUC         : 0.9532
  MCC         : 0.7649
Resultados    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_baselines.parquet
HTML          : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m01_baselines.html

➡️  Siguiente  : fautoml_m02_lazypredict.ipynb
